# Case study

Protein-neighborhood examples and size-adjusted associations with topology-only
prediction scores. Both paper figures are shown below.

See the [setup instructions](../../README.md) for dependencies.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../data/README.md) and
[DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can regenerate
the analysis and figures from completed model runs using the functions in
[`src/dtpkg/case_study/`](../../src/dtpkg/case_study/).

## Paths

Fill in `PROJECT_ROOT` to use the public tables. The other paths are needed only
for the optional private steps. `PREPARED_DIR` must match `INDUCTIVE_RUN`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout
DATA_DIR = ...  # Prepared private data
GRAPH_PATH = ...  # Canonical GraphML network
TRANSDUCTIVE_RUN = ...  # Completed transductive run
INDUCTIVE_RUN = ...  # Completed DDI-free inductive run
PREPARED_DIR = ...  # Drug holdouts used by that inductive run
EXAMPLES_JSON = ...  # Optional local comparative_overlap_examples/examples.json

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

EXPERIMENT = PROJECT_ROOT / "experiments" / "07_case_study"
TABLES = EXPERIMENT / "tables"
OUTPUT_DIR = EXPERIMENT / "figures" / "reproduced"
RESULTS_DIR = EXPERIMENT / "results" / "case_study"

REDRAW_CORRELATIONS = False
REDRAW_NETWORK = False
RUN_ANALYSIS = False
BUILD_EXAMPLES = False

## Protein-neighborhood illustration

![Seen–unseen indirect protein overlap](figures/seen_unseen_indirect_overlap_example.png)

The two DDI-free examples contrast a high-scoring positive and a low-scoring
reliable negative. They share indirect protein context without sharing direct
targets. Selection was post hoc, using scores, overlap and neighborhood sizes.
[Full caption](figures/seen_unseen_indirect_overlap_example_caption.txt).

Set `REDRAW_NETWORK = True` to redraw from your local exported neighborhoods.

In [ ]:
from IPython.display import Image, display
from dtpkg.case_study.render_indirect_example import render_example

if REDRAW_NETWORK:
    if EXAMPLES_JSON is ...:
        raise ValueError("Fill in EXAMPLES_JSON before redrawing the network.")
    paths = render_example(
        Path(EXAMPLES_JSON).expanduser().resolve(),
        OUTPUT_DIR / "seen_unseen_indirect_overlap_example",
    )
    display(Image(filename=str(paths["png"])))

## Adjusted overlap correlations

![Size-adjusted overlap correlations](figures/adjusted_overlap_correlations.png)

Ranked overlap counts and scores are separately adjusted for the ranked smaller
and larger endpoint protein-neighborhood sizes. Their residuals are correlated
without reranking. Dots show repeat or holdout means; diamonds show their mean.
The adjustment is exploratory and split variation is descriptive.
[Full caption](figures/adjusted_overlap_correlations_caption.txt).

The paper snapshot uses 25 transductive fits and 15 DDI-free inductive fits
(five holdouts × three seeds). These holdouts differ from experiment 05.

In [ ]:
from dtpkg.case_study.reporting import load_tables
from dtpkg.case_study.adjusted_overlap_figure import plot_adjusted_overlap

report = load_tables(TABLES)
display(report["adjusted_summary"][["scenario", "feature", "mean_rho", "sd_rho"]])

if REDRAW_CORRELATIONS:
    paths = plot_adjusted_overlap(TABLES, OUTPUT_DIR)
    display(Image(filename=str(paths["png"])))

,scenario,feature,mean_rho,sd_rho
0,transductive,shared_protein_count,0.620195,0.006913
1,transductive,biological_shared_protein_count,0.212272,0.002377
2,seen_unseen,biological_shared_protein_count,0.212782,0.017338
3,unseen_unseen,biological_shared_protein_count,0.177062,0.088819


## Regenerate the analysis

Set `RUN_ANALYSIS = True` to analyse completed transductive and DDI-free inductive
fits using the local network and matching prepared holdouts. This reuses saved
topology-only scores; it does not train models.
[Experiments 04](../04_transductive_fusion/README.md) and
[05](../05_inductive_fusion/README.md) provide training workflows. Using different
fits or holdouts produces new results rather than the saved paper snapshot.

In [ ]:
if RUN_ANALYSIS:
    from dtpkg.case_study.workflow import run_analysis
    from dtpkg.case_study.reporting import export_tables

    private_paths = {
        "DATA_DIR": DATA_DIR, "GRAPH_PATH": GRAPH_PATH,
        "TRANSDUCTIVE_RUN": TRANSDUCTIVE_RUN, "INDUCTIVE_RUN": INDUCTIVE_RUN,
        "PREPARED_DIR": PREPARED_DIR,
    }
    missing = [name for name, value in private_paths.items() if value is ...]
    if missing:
        raise ValueError("Fill in: " + ", ".join(missing))
    private_paths = {name: Path(value).expanduser().resolve()
                     for name, value in private_paths.items()}
    run_analysis("transductive", RESULTS_DIR, source_options={
        "results_dir": private_paths["TRANSDUCTIVE_RUN"],
        "graph_path": private_paths["GRAPH_PATH"],
    })
    run_analysis("inductive", RESULTS_DIR, source_options={
        "results_dir": private_paths["INDUCTIVE_RUN"],
        "prepared_dir": private_paths["PREPARED_DIR"],
        "graph_path": private_paths["GRAPH_PATH"],
        "data_dir": private_paths["DATA_DIR"],
    })
    new_tables = RESULTS_DIR / "public_tables"
    export_tables(RESULTS_DIR, new_tables)
    paths = plot_adjusted_overlap(new_tables, OUTPUT_DIR / "new_run")
    display(Image(filename=str(paths["png"])))

## Rebuild the example neighborhoods

After the analysis, set `BUILD_EXAMPLES = True` to apply the selection rule and
export the complete neighborhoods locally. The selected fit is split 00, seed 101;
selection stops if no pair meets the criteria.

In [ ]:
if BUILD_EXAMPLES:
    from dtpkg.case_study.examples import export_examples

    if any(value is ... for value in (DATA_DIR, GRAPH_PATH, PREPARED_DIR)):
        raise ValueError("Fill in DATA_DIR, GRAPH_PATH and PREPARED_DIR.")
    examples = RESULTS_DIR / "comparative_overlap_examples"
    export_examples(
        RESULTS_DIR, Path(GRAPH_PATH).expanduser().resolve(),
        Path(PREPARED_DIR).expanduser().resolve() / "split_00" / "manifest.json",
        Path(DATA_DIR).expanduser().resolve() / "drugbank" / "drugbank_drugs_release.csv",
        examples,
    )
    paths = render_example(examples / "examples.json",
                           OUTPUT_DIR / "new_run" / "seen_unseen_indirect_overlap_example")
    display(Image(filename=str(paths["png"])))